[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P16/P16-1_Spectre_CORRIGE.ipynb)

# Filtrage et analyse spectrale

Chapitre 16 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre banc d'essai enregistre la vibration d'un moto-réducteur qui tourne à $1\,500$ tr/min, et le tracé en fonction du temps ressemble à du bruit. Vous cherchez la fréquence de rotation, ses harmoniques, et la raie qui n'a rien à y faire, signe d'un défaut naissant ? Vous voudriez isoler une bande, en retirer une autre, et savoir de combien le reste est atténué ?

**Objectif terminal.** Ce chapitre montre comment, avec NumPy et le module `scipy.signal` de SciPy.

**Objectif instrumental.**

- Calculer le spectre d'un signal avec `rfft` et lire fréquences et amplitudes.
- Moyenner les spectres avec `welch` et régler la résolution contre la dispersion.
- Filtrer une bande avec un filtre de Butterworth, sans décalage de phase.
- Repérer des pics par leur proéminence et reconnaître les harmoniques d'une rotation.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3 à 6 et 11, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 16.1 | Les pics d'un signal simulé | 15 min |
| 16.2 | Résolution de Welch | 15 min |
| 16.3 | Un filtre passe-bande à 50 Hz | 15 min |
| 16.4 | Harmoniques d'un défaut (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [1]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("scipy", version("scipy"))

Python 3.12.0
numpy 2.4.6
scipy 1.18.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [2]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [3]:
import math
import numpy as np
from scipy.signal import butter, find_peaks, sosfiltfilt, welch
# Le signal du chapitre : trois composantes et un bruit gaussien, échantillonné à 2 048 Hz pendant 4 s.
FS, DUREE, GRAINE = 2048.0, 4.0, 2026
COMPOSANTES = [(25.0, 2.0), (50.0, 0.8), (87.25, 0.25)]           # (fréquence en Hz, amplitude en mm/s)
BRUIT = 0.3                                                       # écart-type du bruit, mm/s
N = int(FS * DUREE)

### Section 16.1 : un signal se projette sur des sinusoïdes.

In [4]:
# notion: signal-simule
t = np.arange(N) / FS                                             # 8 192 instants, de 0 à 4 s
rng = np.random.default_rng(GRAINE)                               # graine fixée : le même signal à chaque exécution
signal = sum(a * np.sin(2 * math.pi * f * t) for f, a in COMPOSANTES) + rng.normal(0.0, BRUIT, N)
print("points :", N, "; durée :", DUREE, "s ; écart-type du signal :", round(float(signal.std()), 3), "mm/s")
assert (
    signal.shape == (8192,) and abs(t[-1] - (DUREE - 1 / FS)) < 1e-12
), "8 192 points, de 0 à 4 s moins un pas"
assert abs(signal.std() - 1.567) < 0.0005, "écart-type attendu : 1,567 mm/s"
signal_bis = sum(a * np.sin(2 * math.pi * f * t) for f, a in COMPOSANTES) + np.random.default_rng(GRAINE).normal(0.0, BRUIT, N)
assert np.array_equal(signal, signal_bis), "même graine, même signal"

points : 8192 ; durée : 4.0 s ; écart-type du signal : 1.567 mm/s


In [5]:
# notion: rfft
spectre = 2 * np.abs(np.fft.rfft(signal)) / N                      # amplitude de chaque composante, en mm/s
freq_hz = np.fft.rfftfreq(N, d=1 / FS)                          # fréquences associées, de 0 à fs / 2
resolution = float(freq_hz[1] - freq_hz[0])
print("spectre :", spectre.shape, "; fréquence maximale :", freq_hz[-1], "Hz ; résolution :", resolution, "Hz")
assert spectre.shape == freq_hz.shape == (N // 2 + 1,), "N / 2 + 1 fréquences"
assert (
    freq_hz[-1] == FS / 2 and abs(resolution - 1 / DUREE) < 1e-12
), "fréquence de Nyquist, résolution 1 / durée"
# Replier : échantillonné à 100 Hz seulement, un signal à 87,25 Hz apparaît à 12,75 Hz.
fs_lent = 100.0
t_lent = np.arange(int(fs_lent * DUREE)) / fs_lent
lent = np.sin(2 * math.pi * 87.25 * t_lent)
freq_lente = np.fft.rfftfreq(len(t_lent), d=1 / fs_lent)
pic_lent = float(freq_lente[np.argmax(np.abs(np.fft.rfft(lent)))])
print("signal à 87,25 Hz échantillonné à 100 Hz : pic lu à", pic_lent, "Hz")

spectre : (4097,) ; fréquence maximale : 1024.0 Hz ; résolution : 0.25 Hz
signal à 87,25 Hz échantillonné à 100 Hz : pic lu à 12.75 Hz


In [6]:
assert abs(pic_lent - 12.75) < 1e-9, "repliement : 100 - 87,25 = 12,75 Hz"

## Exercice 16.1 : Les pics d'un signal simulé (15 min)

Avec le signal simulé de cette section, aux composantes de $25$, $50$ et $87{,}25$ Hz, lisez les trois pics du spectre : un pic est un point plus haut que ses deux voisins et que le seuil de $0{,}1$ mm/s. Donnez leurs fréquences et leurs amplitudes, et comparez les amplitudes à celles que l'on a injectées.

**Résultat attendu.** Pics à 25,0, 50,0 et 87,25 Hz, d'amplitudes lues 2,008, 0,797 et 0,252 mm/s, contre 2,0, 0,8 et 0,25 mm/s injectées.

**Seconde voie.** La projection sur une sinusoïde d'essai de 50,0 Hz, comme à la section 16.1, redonne l'amplitude du pic à $0{,}02$ mm/s près.

**Indication.** Le milieu du spectre est `spectre[1:-1]` ; on le compare à `spectre[:-2]` et `spectre[2:]`, ses voisins de gauche et de droite, et au seuil, avec `&` entre les trois masques. Les fréquences et les amplitudes sortent par le masque. L'écart aux amplitudes injectées est celui du bruit, de l'ordre de $0{,}01$ mm/s.

Les contrôles de la cellule lisent les variables `a_pics`, `f_pics`, `moyenne_produit` : gardez ces noms.

In [7]:
# Solution :
milieu = spectre[1:-1]
est_pic = (milieu > spectre[:-2]) & (milieu > spectre[2:]) & (milieu > 0.1)
f_pics = freq_hz[1:-1][est_pic]
a_pics = milieu[est_pic]
print("fréquences des pics (Hz) :", f_pics)
print("amplitudes lues (mm/s) :", np.round(a_pics, 3))
# Fin de la solution
assert np.allclose(f_pics, [25.0, 50.0, 87.25]), "pics attendus : 25,0 ; 50,0 ; 87,25 Hz"
assert (
    np.allclose(a_pics, [2.008, 0.797, 0.252], atol=0.0005)
), "amplitudes lues : 2,008 ; 0,797 ; 0,252 mm/s"
assert (
    np.allclose(a_pics, [a for _, a in COMPOSANTES], atol=0.03)
), "elles retrouvent les amplitudes injectées, au bruit près"
# Solution :
moyenne_produit = float(np.mean(signal * np.sin(2 * math.pi * 25.0 * t)))
# Fin de la solution
assert (
    abs(2 * moyenne_produit - a_pics[0]) < 0.02
), "projeter sur une sinusoïde d'essai donne l'amplitude à 25 Hz"
print("Exercice 16.1 : contrôles passés")

fréquences des pics (Hz) : [25.   50.   87.25]
amplitudes lues (mm/s) : [2.008 0.797 0.252]
Exercice 16.1 : contrôles passés


### Section 16.2 : Méthode de Welch.

In [8]:
# notion: welch
f_w, psd = welch(signal, fs=FS, nperseg=1024, scaling="spectrum")     # segments de 1 024 points, recouvrement de moitié
resolution_welch = float(f_w[1] - f_w[0])
print("Welch : segments de 1 024 points ; résolution", resolution_welch, "Hz contre", resolution, "Hz")
assert (
    abs(resolution_welch - FS / 1024) < 1e-12 and resolution_welch == 2.0
), "2 Hz : huit fois moins fin"
puissance = (np.abs(np.fft.rfft(signal))**2) * 2 / N**2 * 2          # même grandeur que scaling="spectrum" sans fenêtre
plancher = (freq_hz > 200.0) & (freq_hz < 1000.0)              # une bande sans composante : du bruit seul
plancher_w = (f_w > 200.0) & (f_w < 1000.0)
dispersion_brute = float(np.std(puissance[plancher]) / np.mean(puissance[plancher]))
dispersion_welch = float(np.std(psd[plancher_w]) / np.mean(psd[plancher_w]))
print(f"dispersion relative du plancher de bruit : brut {dispersion_brute:.2f} ; Welch {dispersion_welch:.2f}")
assert dispersion_brute > 0.9, "un spectre brut fluctue autant que sa moyenne"
assert dispersion_welch < 0.4, "moyenner sur les segments divise la fluctuation"

Welch : segments de 1 024 points ; résolution 2.0 Hz contre 0.25 Hz
dispersion relative du plancher de bruit : brut 0.99 ; Welch 0.25


## Exercice 16.2 : Résolution de Welch (15 min)

Sur le signal de la section précédente, calculez le spectre de Welch avec des segments de $1\,024$ points. Donnez sa résolution, comparez à celle du spectre brut, mesurez la dispersion du plancher de bruit pour le spectre brut et pour Welch, et lisez la rotation à $25$ Hz.

**Résultat attendu.** Résolution de 2 Hz contre 0,25 Hz ; dispersion de 0,99 pour le spectre brut et 0,25 pour Welch, soit une fluctuation divisée par 3,9 ; rotation lue à 26 Hz.

**Seconde voie.** Le nombre de segments de recouvrement moitié est $(N - n/2)/(n/2)$, soit $15$ ici ; sa racine est du même ordre que le rapport des dispersions.

**Indication.** On appelle `welch` avec le réglage `nperseg=1024` ; la fonction rend les fréquences et le spectre ; la résolution est `f_w[1] - f_w[0]`. Pour le plancher, on garde les cases entre $200$ et $1\,000$ Hz et l'on calcule écart-type sur moyenne, pour le carré du spectre brut et pour le spectre de Welch. La rotation est le maximum de la zone de $20$ à $30$ Hz.

Les contrôles de la cellule lisent les variables `f_25_welch`, `rapport_dispersion` : gardez ces noms.

In [9]:
# Solution :
rapport_dispersion = dispersion_brute / dispersion_welch
indice_25 = int(np.argmax(psd[(f_w > 20.0) & (f_w < 30.0)])) + int(np.argmax(f_w > 20.0))
f_25_welch = float(f_w[indice_25])
print(f"résolution : {resolution} Hz brute, {resolution_welch} Hz avec Welch ; fluctuation divisée par {rapport_dispersion:.1f}")
print("pic proche de 25 Hz lu à", f_25_welch, "Hz avec Welch")
# Fin de la solution
assert resolution == 0.25 and resolution_welch == 2.0, "0,25 Hz contre 2 Hz"
assert rapport_dispersion > 2.5, "la fluctuation est divisée par plus de deux et demi"
assert (
    abs(f_25_welch - 25.0) <= resolution_welch / 2 + 1e-12
), "le pic de 25 Hz est lu à la résolution près"
print("Exercice 16.2 : contrôles passés")

résolution : 0.25 Hz brute, 2.0 Hz avec Welch ; fluctuation divisée par 3.9
pic proche de 25 Hz lu à 26.0 Hz avec Welch
Exercice 16.2 : contrôles passés


### Section 16.3 : un filtre retire une bande.

In [10]:
# notion: butter
# Passe-bande d'ordre 4 de 40 à 60 Hz, sous forme de sections du second ordre ; sosfiltfilt filtre dans les deux sens,
# donc sans décalage de phase (et avec un gain élevé au carré).
sos = butter(4, [40.0, 60.0], btype="bandpass", fs=FS, output="sos")
filtre = sosfiltfilt(sos, signal)
spectre_filtre = 2 * np.abs(np.fft.rfft(filtre)) / N

def amplitude_a(spectre_x, f_hz):
    """Amplitude lue dans un spectre à la fréquence f_hz (la plus proche de la grille)."""
    return float(spectre_x[int(round(f_hz / resolution))])

print("sections du second ordre :", sos.shape)
print("amplitude à 50 Hz après filtrage :", round(amplitude_a(spectre_filtre, 50.0), 3), "mm/s")
assert sos.shape == (4, 6), "ordre 4 en passe-bande : quatre sections du second ordre"
assert (
    abs(amplitude_a(spectre_filtre, 50.0) - amplitude_a(spectre, 50.0)) < 0.02
), "la bande passante garde le 50 Hz"

sections du second ordre : (4, 6)
amplitude à 50 Hz après filtrage : 0.791 mm/s


In [11]:
assert (
    amplitude_a(spectre_filtre, 25.0) < 0.05 * amplitude_a(spectre, 25.0)
), "le 25 Hz est supprimé"

## Exercice 16.3 : Un filtre passe-bande à 50 Hz (15 min)

Sur le signal d'essai du chapitre, aux composantes de $25$, $50$ et $87{,}25$ Hz, filtrez avec un passe-bande de Butterworth d'ordre $4$ de $40$ à $60$ Hz. Lisez l'amplitude à $50$ Hz après filtrage, l'atténuation de la rotation à $25$ Hz en décibels, et la valeur efficace du signal avant et après filtrage.

**Résultat attendu.** Amplitude de 0,791 mm/s à 50,0 Hz ; rotation atténuée de 61 dB ; valeur efficace de 1,567, puis 0,562 mm/s.

**Seconde voie.** Il ne reste que la composante à 50,0 Hz : sa valeur efficace est son amplitude divisée par $\sqrt{2}$.

**Indication.** La fonction `butter`, d'ordre $4$, de coupures $40$ et $60$ Hz, avec `btype="bandpass"` et `output="sos"`, donne quatre sections ; on filtre avec `sosfiltfilt`. On relit le spectre du signal filtré à 50,0 et $25$ Hz, et l'atténuation est $20\log_{10}$ du rapport de l'amplitude d'origine à l'amplitude filtrée. La valeur efficace est `np.sqrt(np.mean(x**2))`.

Les contrôles de la cellule lisent les variables `amplitude_50`, `att_dB`, `rms_apres` : gardez ces noms.

In [12]:
# Solution :
amplitude_50 = amplitude_a(spectre_filtre, 50.0)
attenuation_25 = amplitude_a(spectre_filtre, 25.0) / amplitude_a(spectre, 25.0)
att_dB = 20 * math.log10(attenuation_25)
rms_avant, rms_apres = float(np.sqrt(np.mean(signal**2))), float(np.sqrt(np.mean(filtre**2)))
print(f"amplitude à 50 Hz : {amplitude_50:.3f} mm/s ; 25 Hz atténué de {-att_dB:.0f} dB ; valeur efficace {rms_avant:.3f} puis {rms_apres:.3f} mm/s")
# Fin de la solution
assert abs(amplitude_50 - 0.791) < 0.0005, "amplitude filtrée attendue : 0,791 mm/s"
assert -att_dB > 40, "atténuation du 25 Hz supérieure à 40 dB"
assert abs(rms_apres - 0.8 / math.sqrt(2)) < 0.01, "il reste le 50 Hz : amplitude / racine(2)"
print("Exercice 16.3 : contrôles passés")

amplitude à 50 Hz : 0.791 mm/s ; 25 Hz atténué de 61 dB ; valeur efficace 1.567 puis 0.562 mm/s
Exercice 16.3 : contrôles passés


### Section 16.4 : les pics se repèrent par leur proéminence.

In [13]:
# notion: find-peaks
indices, proprietes = find_peaks(spectre, prominence=0.1)          # la proéminence : la hauteur au-dessus de la base du pic
pics_fins, _ = find_peaks(spectre, prominence=0.001)
print("pics de proéminence au moins 0,1 mm/s :", freq_hz[indices], "; au moins 0,001 mm/s :", len(pics_fins))
assert np.allclose(freq_hz[indices], [25.0, 50.0, 87.25]), "trois pics"
assert (
    np.all(proprietes["prominences"] > 0.1) and len(pics_fins) > 100
), "un seuil trop bas retient du bruit"

pics de proéminence au moins 0,1 mm/s : [25.   50.   87.25] ; au moins 0,001 mm/s : 1202


## Exercice 16.4 : Harmoniques d'un défaut (défi, 25 min)

Reprenez les pics de proéminence au moins $0{,}1$ mm/s du signal de ce chapitre. Donnez le rang de chacun par rapport à la rotation de $25$ Hz, indiquez lesquels sont des harmoniques, et nommez la composante de défaut avec son rang. Que devient la conclusion si l'on retient, à la place, tous les pics de proéminence au moins $0{,}001$ mm/s ?

**Résultat attendu.** Rangs de $1$, $2$ et 3,49 ; harmoniques de rang $1$ et $2$ ; composante de défaut à 87,25 Hz ; avec le seuil de $0{,}001$ mm/s, 1 202 pics, et presque tous sont du bruit.

**Seconde voie.** La projection sur une sinusoïde d'essai de 87,25 Hz redonne une amplitude proche de 0,25 mm/s, alors que celle de $87{,}5$ Hz, un quart de hertz plus haut, ne rend presque rien.

**Indication.** Les rangs sont `freq[indices] / 25.0` ; un rang est un harmonique si son écart à l'entier le plus proche, `abs(rang - round(rang))`, est sous $0{,}02$. Les rangs $1$ et $2$ le sont, le troisième non : sa fréquence est 87,25 Hz. Avec le seuil de $0{,}001$ mm/s, la liste contient plus de mille pics, presque tous issus du bruit : le seuil de proéminence est un réglage à justifier.

Les contrôles de la cellule lisent les variables `f_defaut`, `harmoniques`, `rangs` : gardez ces noms.

In [14]:
f_rotation = 25.0
# Solution :
rangs = freq_hz[indices] / f_rotation
harmoniques = [bool(abs(r - round(r)) < 0.02) for r in rangs]
f_defaut = float(freq_hz[indices][[not h for h in harmoniques].index(True)])
print("rapports à la rotation :", np.round(rangs, 2), "; harmoniques :", harmoniques, "; défaut à", f_defaut, "Hz")
# Fin de la solution
assert (
    harmoniques == [True, True, False]
), "25 et 50 Hz sont les rangs 1 et 2 ; 87,25 Hz n'est pas un multiple"
assert abs(f_defaut - 87.25) < 1e-9, "la composante de défaut est à 87,25 Hz"
assert abs(rangs[2] - 3.49) < 0.005, "rapport de 87,25 Hz à la rotation : 3,49"
print("Exercice 16.4 : contrôles passés")

rapports à la rotation : [1.   2.   3.49] ; harmoniques : [True, True, False] ; défaut à 87.25 Hz
Exercice 16.4 : contrôles passés


## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F16-1 : Spectre d'un signal (`numpy.fft.rfft`)

In [15]:
import numpy as np

fs_b, tb = 1000.0, np.arange(1000) / 1000.0   # 1 seconde à 1 000 Hz
vib = 2.0 * np.sin(2 * np.pi * 50 * tb) + 5.0 * np.cos(2 * np.pi * 100 * tb)
X_b = np.fft.rfft(vib)                    # transformée complexe
amp = 2 * np.abs(X_b) / vib.size          # amplitudes, m/s²
freq_b = np.fft.rfftfreq(vib.size, 1/fs_b)   # axe des fréquences, Hz
k_max = np.argmax(amp)                    # indice du pic principal
msg = f"pics à 50 et 100 Hz : {amp[50]:.1f} et {amp[100]:.1f} m/s²"
print(msg.replace(".", ","))
print(np.isclose(freq_b[k_max], 100.0))     # l'allumage domine

pics à 50 et 100 Hz : 2,0 et 5,0 m/s²
True


### Fiche F16-2 : Filtrage numérique (`scipy.signal.sosfiltfilt`)

In [16]:
import numpy as np
from scipy.signal import butter, sosfiltfilt

t_t = np.arange(400) / 200.0            # 2 s d'essai à 200 Hz
x_t = 5000.0 + 200.0 * np.sin(np.pi * t_t) + 80.0 * np.sin(24.0 * np.pi * t_t)
sos_t = butter(4, [8.0, 16.0], btype="bandpass", fs=200.0, output="sos")
x_filtre = sosfiltfilt(sos_t, x_t)
ptp = np.ptp(x_filtre[50:-50])          # étendue utile sans bords

x_moy, f_moy = np.mean(x_t), np.mean(x_filtre)
print(f"Moyennes : {x_moy:.0f} puis {f_moy:.1f} N".replace(".", ","))
print(f"Étendue utile : {ptp:.1f} N".replace(".", ","))
assert np.abs(f_moy) < 1.0              # moyenne effacée

Moyennes : 5000 puis 0,2 N
Étendue utile : 160,8 N
